# GUS04E — Deep Data Audit & Hierarchical Consistency

This notebook performs a comprehensive audit of the GeoTERYT database to identify
all data issues that could cause problems during demographic estimation.

**Issues investigated:**
1. Missing M_ merged subjects for new voivodeships
2. Shape inconsistencies in merged subjects
3. Warsaw (1431) double-counting in hierarchy
4. Hierarchical consistency: gminas → powiats → voivodeships for ALL years
5. M_pop__age_sex vs M_age_sex label discrepancies
6. Root cause analysis in `_create_custom_merged_subjects()`

In [1]:
# ── Cell 1: Imports & load database ──
import sys, os, time
import numpy as np
import pandas as pd
from collections import Counter, defaultdict

REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, os.path.join(REPO, 'Code', 'tools'))
DATA_ROOT = os.path.join(REPO, '..', '..', 'Data', 'Geospatial')

from geoTERYT_db import (
    load_complete_database, LEVEL_GMINA, LEVEL_VOIVODESHIP, LEVEL_POWIAT,
)

db_path = os.path.join(DATA_ROOT, 'geoteryt_O.pkl')
print(f"Loading database from {db_path}…")
t0 = time.time()
db = load_complete_database(db_path)
print(f"Loaded in {time.time()-t0:.1f}s — {len(db._records)} records")

Loading database from /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper/../../Data/Geospatial/geoteryt_O.pkl…
Loading complete database from /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper/../../Data/Geospatial/geoteryt_O.pkl...
  Database version: 4.3
  ✓ Restored old voivodships: 49 rows
  ✓ Restored geometry store: 13,287 unique geometries
  ✓ Restored geometry data for years: [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2015, 2016, 2017, 2018, 2021, 2022, 2023]
  ✓ Loaded 4612 records
  ✓ Year range: 1999 - 2024
  ✓ Records with geometry: 3661
  ✓ Records with old_woj: 4104
  ✓ Records with data: 4584
  ✓ Records with cross tables: 4584
  ✓ Records with population data: 4582
  ✓ Records with pop_class: 3411
Loaded in 380.3s — 4612 records


## 1. Subject Inventory — What merged subjects exist?

In [2]:
# ── Cell 2: Cross table inventory ──
all_ct = Counter()
ct_lvl = defaultdict(set)
ct_shp = defaultdict(set)
ct_yrs = defaultdict(set)

for rec in db._records.values():
    for sid, ct in rec.cross_tables.items():
        all_ct[sid] += 1
        ct_lvl[sid].add(rec.level)
        ct_shp[sid].add(ct.shape)
        ct_yrs[sid].update(ct.years_with_data)

print(f"Total unique cross table subjects: {len(all_ct)}\n")
print("M_, H_, E_ subjects:")
print(f"{'Subject':<35} {'Records':>7} {'Levels':<20} {'Shapes'}")
print("-" * 90)
for sid in sorted(all_ct):
    if sid.startswith(('M_', 'H_', 'E_')):
        yrs = sorted(ct_yrs[sid])
        yr_str = f"{min(yrs)}-{max(yrs)}" if yrs else "none"
        print(f"{sid:<35} {all_ct[sid]:>7} {str(sorted(ct_lvl[sid])):<20} {ct_shp[sid]}  [{yr_str}]")

# Needed subjects check
needed = ['M_age_sex', 'M_age_1990', 'M_educ_1990', 'M_educ_2000', 
          'M_educ_sex_1990', 'M_educ_sex_2000', 'M_hh_size_1990', 'M_hh_size_2000']
print(f"\n{'='*60}")
print("Needed v5.0 subjects:")
for sid in needed:
    ct_count = all_ct.get(sid, 0)
    shapes = ct_shp.get(sid, set())
    status = '✓' if ct_count > 0 else '✗ MISSING'
    shape_issue = ' ⚠ INCONSISTENT SHAPES' if len(shapes) > 1 else ''
    print(f"  {sid:<30} {status}  recs={ct_count}  shapes={shapes}{shape_issue}")

Total unique cross table subjects: 35

M_, H_, E_ subjects:
Subject                             Records Levels               Shapes
------------------------------------------------------------------------------------------
H_age_sex                                50 [0, 2]               {(17, 3)}  [1986-1994]
H_educ_age                                1 [0]                  {(3, 5, 9)}  [1988-1988]
H_sex_educ                                1 [0]                  {(3, 6)}  [1986-1994]
M_age_1990                             4582 [0, 2, 5, 6]         {(8,)}  [1986-2024]
M_age_sex                              4582 [0, 2, 5, 6]         {(16, 3)}  [1986-2024]
M_educ_1990                            3725 [0, 6]               {(6,)}  [1986-2002]
M_educ_2000                            4276 [0, 2, 5, 6]         {(5,)}  [1995-2024]
M_educ_sex_1990                        3648 [0, 6]               {(6, 3)}  [1986-2002]
M_educ_sex_2000                        4257 [5, 6]               {(5, 3)}  [2002-2

## 2. M_age_sex — New voivodeship gap

In [3]:
# ── Cell 3: Check M_age_sex availability per level ──
# New voivodeships = teryt_id < 5000000 AND level=2 (excludes old 49 voivodeships)
new_voi = sorted([r.teryt_id for r in db._records.values()
                  if r.level == LEVEL_VOIVODESHIP and int(r.teryt_id) < 5000000])
old_voi = sorted([r.teryt_id for r in db._records.values()
                  if r.level == LEVEL_VOIVODESHIP and int(r.teryt_id) >= 5000000])

print(f"New voivodeships: {len(new_voi)}, Old voivodeships: {len(old_voi)}")
print(f"\n{'='*80}")
print(f"{'TID':<10} {'Name':<25} {'M_age_sex':>10} {'M_pop__as':>10} {'M_educ00':>10} {'M_educ90':>10} {'M_hh00':>8}")
print("-" * 80)

subjects_checked = ['M_age_sex', 'M_pop__age_sex', 'M_educ_2000', 'M_educ_1990', 'M_hh_size_2000']

for tid in new_voi:
    rec = db.get_by_teryt_id(tid)
    vals = []
    for sid in subjects_checked:
        ct = rec.cross_tables.get(sid)
        if ct:
            n = len(ct.years_with_data)
            vals.append(f"{n:>4}yr")
        else:
            vals.append(f"{'---':>6}")
    print(f"{tid:<10} {rec.name:<25} {vals[0]:>10} {vals[1]:>10} {vals[2]:>10} {vals[3]:>10} {vals[4]:>8}")

print(f"\n{'='*80}")
print("Old voivodeships (sample):")
for tid in old_voi[:5]:
    rec = db.get_by_teryt_id(tid)
    vals = []
    for sid in subjects_checked:
        ct = rec.cross_tables.get(sid)
        if ct:
            n = len(ct.years_with_data)
            vals.append(f"{n:>4}yr")
        else:
            vals.append(f"{'---':>6}")
    print(f"{tid:<10} {rec.name:<25} {vals[0]:>10} {vals[1]:>10} {vals[2]:>10} {vals[3]:>10} {vals[4]:>8}")

# Check coverage
n_with = sum(1 for tid in new_voi if db.get_by_teryt_id(tid).cross_tables.get('M_age_sex'))
n_total = len(new_voi)
if n_with == n_total:
    print(f"\n✓ FIX VERIFIED: M_age_sex available for ALL {n_total} new voivodeships")
elif n_with >= 16:
    missing_tids = [tid for tid in new_voi if not db.get_by_teryt_id(tid).cross_tables.get('M_age_sex')]
    missing_names = [db.get_by_teryt_id(t).name for t in missing_tids]
    print(f"\n✓ FIX VERIFIED: M_age_sex for {n_with}/{n_total} new voivodeships (16 standard + old)")
    print(f"  Missing ({len(missing_tids)}): {', '.join(f'{t} {n}' for t, n in zip(missing_tids, missing_names))}")
    print(f"  (These are NUTS-2 subregions of Mazowieckie, not standard voivodeships)")
else:
    print(f"\n⚠ ISSUE: M_age_sex only for {n_with}/{n_total} new voivodeships")

New voivodeships: 18, Old voivodeships: 49

TID        Name                       M_age_sex  M_pop__as   M_educ00   M_educ90   M_hh00
--------------------------------------------------------------------------------
0200000    DOLNOŚLĄSKIE                    30yr       30yr       30yr        ---      ---
0400000    KUJAWSKO-POMORSKIE              30yr       30yr       30yr        ---      ---
0600000    LUBELSKIE                       30yr       30yr       30yr        ---      ---
0800000    LUBUSKIE                        30yr       30yr       30yr        ---      ---
1000000    ŁÓDZKIE                         30yr       30yr       30yr        ---      ---
1200000    MAŁOPOLSKIE                     30yr       30yr       30yr        ---      ---
1300000    Warszawski stołeczny             ---        ---       24yr        ---      ---
1400000    MAZOWIECKIE                     30yr       30yr       30yr        ---      ---
1500000    Mazowiecki regionalny            ---        ---       

## 3. Shape Inconsistencies

In [4]:
# ── Cell 4: Investigate shape inconsistencies ──

# --- M_pop__age_sex: (19,3) vs (16,3) ---
print("=" * 80)
print("M_pop__age_sex shape inconsistency")
print("=" * 80)
mpas_shp_by_level = defaultdict(Counter)
mpas_16_tids = []
for rec in db._records.values():
    ct = rec.cross_tables.get('M_pop__age_sex')
    if ct:
        mpas_shp_by_level[rec.level][ct.shape] += 1
        if ct.shape == (16, 3):
            mpas_16_tids.append((rec.teryt_id, rec.name, rec.level, rec.rodz))

for lvl in sorted(mpas_shp_by_level):
    print(f"  level={lvl}: {dict(mpas_shp_by_level[lvl])}")

print(f"\nRecords with (16,3) shape ({len(mpas_16_tids)}):")
for tid, name, lvl, rodz in mpas_16_tids[:20]:
    print(f"  {tid} rdz={rodz} lvl={lvl} {name}")
if len(mpas_16_tids) > 20:
    print(f"  ... ({len(mpas_16_tids)-20} more)")

# --- M_educ_1990: (6,) vs (5,) ---
print(f"\n{'='*80}")
print("M_educ_1990 shape inconsistency")
print("=" * 80)
for rec in db._records.values():
    ct = rec.cross_tables.get('M_educ_1990')
    if ct and ct.shape == (5,):
        print(f"  {rec.teryt_id} {rec.name} shape={ct.shape}")
        print(f"    labels: {ct.dim_labels}")
        print(f"    years with data: {sorted(ct.years_with_data)}")

# --- M_hh_size_2000: (6,) vs (5,) ---
print(f"\n{'='*80}")
print("M_hh_size_2000 shape inconsistency")
print("=" * 80)
hh_wrong = []
hh_wrong_labels = None
for rec in db._records.values():
    ct = rec.cross_tables.get('M_hh_size_2000')
    if ct and ct.shape == (5,):
        hh_wrong.append((rec.teryt_id, rec.name, rec.level))
        if hh_wrong_labels is None:
            hh_wrong_labels = ct.dim_labels

print(f"  Records with (5,) instead of (6,): {len(hh_wrong)}")
print(f"  Labels: {hh_wrong_labels}")
# Check: which label is missing?
ct_ok = None
for rec in db._records.values():
    ct = rec.cross_tables.get('M_hh_size_2000')
    if ct and ct.shape == (6,):
        ct_ok = ct
        break
if ct_ok and hh_wrong_labels:
    ok_labels = set(ct_ok.dim_labels['n1'])
    bad_labels = set(hh_wrong_labels['n1'])
    missing = ok_labels - bad_labels
    print(f"  Missing label: {missing}")
print(f"  Sample records: {hh_wrong[:10]}")

M_pop__age_sex shape inconsistency
  level=0: {(19, 3): 1}
  level=2: {(19, 3): 16}
  level=5: {(19, 3): 381, (16, 3): 1}
  level=6: {(19, 3): 4057, (16, 3): 77}

Records with (16,3) shape (78):
  0220022 rdz=2 lvl=6 Prusice
  0618122 rdz=2 lvl=6 Tyszowce
  0804093 rdz=3 lvl=6 Sława
  0804094 rdz=4 lvl=6 Sława
  0804095 rdz=5 lvl=6 Sława
  0804103 rdz=3 lvl=6 Szlichtyngowa
  0804104 rdz=4 lvl=6 Szlichtyngowa
  0804105 rdz=5 lvl=6 Szlichtyngowa
  0804113 rdz=3 lvl=6 Wschowa
  0804114 rdz=4 lvl=6 Wschowa
  0804115 rdz=5 lvl=6 Wschowa
  1006011 rdz=1 lvl=6 Brzeziny
  1006042 rdz=2 lvl=6 Brzeziny
  1006052 rdz=2 lvl=6 Dmosin
  1006062 rdz=2 lvl=6 Jeżów
  1006092 rdz=2 lvl=6 Rogów
  1202032 rdz=2 lvl=6 Czchów
  1212021 rdz=1 lvl=6 Sławków
  1216062 rdz=2 lvl=6 Ryglice
  1412021 rdz=1 lvl=6 Sulejówek
  ... (58 more)

M_educ_1990 shape inconsistency

M_hh_size_2000 shape inconsistency
  Records with (5,) instead of (6,): 0
  Labels: None
  Sample records: []


## 4. Warsaw Double-Counting

In [5]:
# ── Cell 5: Warsaw (powiat 1431) hierarchy investigation ──
powiat_waw = db.get_by_teryt_id('1431000')

print("Warsaw powiat (1431000) children across years:")
print(f"{'Year':>6} {'n_r123':>7} {'n_waw':>6} {'sum_pop':>12} {'pow_pop':>12} {'ratio':>8}")
print("-" * 55)
for yr in range(1999, 2025):
    ch = powiat_waw.get_children(yr)
    r123 = [g for g in ch if db.get_by_teryt_id(g) and db.get_by_teryt_id(g).rodz in ('1','2','3')]
    ww = [g for g in r123 if g.startswith('1431')]
    ts = pd.Timestamp(yr, 1, 1)
    s = sum(db.get_by_teryt_id(g).pop.get(ts, 0) for g in r123)
    pp = powiat_waw.pop.get(ts, np.nan) if powiat_waw else np.nan
    ratio = s / pp if pp and not np.isnan(pp) and pp > 0 else np.nan
    if not np.isnan(ratio):
        flag = ' ⚠ DOUBLE-COUNT' if ratio > 1.5 else ''
        print(f"{yr:>6} {len(r123):>7} {len(ww):>6} {s:>12.0f} {pp:>12.0f} {ratio:>8.3f}{flag}")

# Show the Warsaw sub-units for year 2000
print(f"\nWarsaw powiat children (rodz 1/2/3) for year 2000:")
ch = powiat_waw.get_children(2000)
for g in sorted(ch):
    gr = db.get_by_teryt_id(g)
    if gr and gr.rodz in ('1','2','3') and g.startswith('1431'):
        pop = gr.pop.get(pd.Timestamp(2000,1,1), np.nan)
        print(f"  {g} rdz={gr.rodz} {gr.name:<30} pop={pop:>12.0f}  yrs={min(gr.years_valid)}-{max(gr.years_valid)}")

# Most important: show that Warszawa pop = sum of district pops
waw_main = db.get_by_teryt_id('1431001')
waw_pop = waw_main.pop.get(pd.Timestamp(2000,1,1), np.nan)
district_tids = ['1431011','1431021','1431031','1431041','1431121','1431131',
                 '1431141','1431151','1431161','1431171','1431181']
district_sum = sum(db.get_by_teryt_id(t).pop.get(pd.Timestamp(2000,1,1), 0) for t in district_tids)
print(f"\n  Warszawa (1431001) pop = {waw_pop:.0f}")
print(f"  Sum of 11 districts    = {district_sum:.0f}")
print(f"  Ratio: {district_sum/waw_pop:.6f}")
print(f"\n  ⚠ CONCLUSION: Warszawa (1431001) population = sum of its 11 districts.")
print(f"    Both parent and children have rodz=1, causing EXACT 2× double-counting")
print(f"    when aggregating rodz 1/2/3 within the powiat.")

Warsaw powiat (1431000) children across years:
  Year  n_r123  n_waw      sum_pop      pow_pop    ratio
-------------------------------------------------------
  1999      12     12      3354632      1677316    2.000 ⚠ DOUBLE-COUNT
  2000      12     12      3344836      1672418    2.000 ⚠ DOUBLE-COUNT
  2001      12     12      3343454      1671727    2.000 ⚠ DOUBLE-COUNT

Warsaw powiat children (rodz 1/2/3) for year 2000:
  1431001 rdz=1 Warszawa                       pop=     1672418  yrs=1999-2024
  1431011 rdz=1 Warszawa-Bemowo                pop=      103746  yrs=1999-2001
  1431021 rdz=1 Warszawa-Białołęka             pop=       52454  yrs=1999-2001
  1431031 rdz=1 Warszawa-Bielany               pop=      139337  yrs=1999-2001
  1431041 rdz=1 Warszawa-Centrum               pop=      945165  yrs=1999-2001
  1431121 rdz=1 Warszawa-Rembertów             pop=       21294  yrs=1999-2001
  1431131 rdz=1 Warszawa-Targówek              pop=      125251  yrs=1999-2001
  1431141 rdz=1 War

## 5. Hierarchical Consistency: Gminas → Powiats → Voivodeships

In [6]:
# ── Cell 6: Population hierarchy check for ALL years ──
# For each year, check if sum of gmina pop (rodz 1/2/3) = powiat pop = voivodeship pop

RODZ_AGG = {'1', '2', '3'}

def check_hierarchy_pop(db, year):
    """Check hierarchical population consistency for a given year."""
    ts = pd.Timestamp(year, 1, 1)
    errors = []
    
    # Check powiats: sum of gmina children (rodz 1/2/3) vs powiat pop
    for rec in db._records.values():
        if rec.level != LEVEL_POWIAT:
            continue
        pow_pop = rec.pop.get(ts, np.nan)
        if np.isnan(pow_pop) or pow_pop == 0:
            continue
        
        children = rec.get_children(year)
        gmina_sum = 0.0
        n_gminas = 0
        for g in children:
            gr = db.get_by_teryt_id(g)
            if gr and gr.rodz in RODZ_AGG:
                gpop = gr.pop.get(ts, np.nan)
                if not np.isnan(gpop):
                    gmina_sum += gpop
                    n_gminas += 1
        
        if n_gminas > 0:
            pct_err = abs(gmina_sum - pow_pop) / pow_pop * 100
            if pct_err > 0.01:  # > 0.01% error
                errors.append({
                    'type': 'powiat',
                    'tid': rec.teryt_id,
                    'name': rec.name,
                    'expected': pow_pop,
                    'actual': gmina_sum,
                    'pct_err': pct_err,
                    'n_children': n_gminas,
                    'ratio': gmina_sum / pow_pop
                })
    
    # Check voivodeships: sum of powiat children vs voiv pop
    for rec in db._records.values():
        if rec.level != LEVEL_VOIVODESHIP or int(rec.teryt_id) >= 5000000:
            continue  # skip old voivodeships
        voi_pop = rec.pop.get(ts, np.nan)
        if np.isnan(voi_pop) or voi_pop == 0:
            continue
        
        children = rec.get_children(year)
        child_sum = 0.0
        n_children = 0
        for c in children:
            cr = db.get_by_teryt_id(c)
            if cr:
                cpop = cr.pop.get(ts, np.nan)
                if not np.isnan(cpop):
                    child_sum += cpop
                    n_children += 1
        
        if n_children > 0:
            pct_err = abs(child_sum - voi_pop) / voi_pop * 100
            if pct_err > 0.01:
                errors.append({
                    'type': 'voivodeship',
                    'tid': rec.teryt_id,
                    'name': rec.name,
                    'expected': voi_pop,
                    'actual': child_sum,
                    'pct_err': pct_err,
                    'n_children': n_children,
                    'ratio': child_sum / voi_pop
                })
    
    return errors

# Run for all years 1999-2024
all_errors = {}
for yr in range(1999, 2025):
    errs = check_hierarchy_pop(db, yr)
    if errs:
        all_errors[yr] = errs

# Summary
print("Population hierarchy errors (gminas→powiats, powiats→voivodeships):")
print(f"{'Year':>6} {'Powiat errs':>12} {'Voiv errs':>10} {'Worst':>40} {'%err':>8}")
print("-" * 80)
for yr in range(1999, 2025):
    errs = all_errors.get(yr, [])
    pow_errs = [e for e in errs if e['type'] == 'powiat']
    voi_errs = [e for e in errs if e['type'] == 'voivodeship']
    if errs:
        worst = max(errs, key=lambda e: e['pct_err'])
        print(f"{yr:>6} {len(pow_errs):>12} {len(voi_errs):>10} {worst['tid']+' '+worst['name']:>40} {worst['pct_err']:>7.2f}%")
    else:
        print(f"{yr:>6} {0:>12} {0:>10} {'(all consistent)':>40}")

# Show detail for years with errors
if all_errors:
    worst_year = max(all_errors.keys(), key=lambda y: max(e['pct_err'] for e in all_errors[y]))
    print(f"\nWorst year detail ({worst_year}):")
    for e in sorted(all_errors[worst_year], key=lambda x: -x['pct_err'])[:10]:
        print(f"  {e['type']:<12} {e['tid']} {e['name']:<25} expected={e['expected']:.0f} got={e['actual']:.0f} ratio={e['ratio']:.4f} err={e['pct_err']:.2f}%")

Population hierarchy errors (gminas→powiats, powiats→voivodeships):
  Year  Powiat errs  Voiv errs                                    Worst     %err
--------------------------------------------------------------------------------
  1999            1          1                       1431000 warszawski  100.00%
  2000            1          1                       1431000 warszawski  100.00%
  2001            1          1                       1431000 warszawski  100.00%
  2002            0          0                         (all consistent)
  2003            0          0                         (all consistent)
  2004            0          0                         (all consistent)
  2005            0          0                         (all consistent)
  2006            0          0                         (all consistent)
  2007            0          0                         (all consistent)
  2008            0          0                         (all consistent)
  2009            0    

## 6. Cross-table (M_age_sex) Hierarchical Consistency

In [7]:
# ── Cell 7: Check M_age_sex cross-table hierarchical consistency ──
# For each voivodeship and year, sum M_age_sex of gminas (rodz 1/2/3)
# and compare with the voivodeship M_pop__age_sex (or M_age_sex if available)

RODZ_AGG = {'1', '2', '3'}
SID = 'M_age_sex'

# Get the standard shape and labels
sample_ct = None
for rec in db._records.values():
    ct = rec.cross_tables.get(SID)
    if ct:
        sample_ct = ct
        break

target_shape = sample_ct.shape
og_age = sample_ct.dim_labels['n1'].index('ogółem')
og_sex = sample_ct.dim_labels['n2'].index('ogółem')
print(f"M_age_sex: shape={target_shape}, ogółem indices: age={og_age}, sex={og_sex}")
print(f"Labels n1: {sample_ct.dim_labels['n1']}")

# For each new voivodeship, aggregate gminas and compare with voiv data
new_voi = sorted([r.teryt_id for r in db._records.values()
                  if r.level == LEVEL_VOIVODESHIP and int(r.teryt_id) < 5000000
                  and r.teryt_id not in ('1300000', '1500000')])  # exclude NUTS sub-regions

print(f"\nChecking {len(new_voi)} new voivodeships × years 1999-2024...")
errors = []
for voiv_tid in new_voi:
    voiv_rec = db.get_by_teryt_id(voiv_tid)
    for yr in range(1999, 2025):
        # Aggregate gminas via powiats
        agg = np.zeros(target_shape)
        n_gminas = 0
        n_skipped = 0
        
        for pow_tid in voiv_rec.get_children(yr):
            pow_rec = db.get_by_teryt_id(pow_tid)
            if not pow_rec or pow_rec.level != LEVEL_POWIAT:
                continue
            for g_tid in pow_rec.get_children(yr):
                g_rec = db.get_by_teryt_id(g_tid)
                if not g_rec or g_rec.rodz not in RODZ_AGG:
                    continue
                gct = g_rec.cross_tables.get(SID)
                if not gct:
                    n_skipped += 1
                    continue
                gtbl = gct.get_table(yr)
                if gtbl is None or np.all(np.isnan(gtbl)):
                    n_skipped += 1
                    continue
                if gtbl.shape != target_shape:
                    n_skipped += 1
                    continue
                agg += np.nan_to_num(gtbl)
                n_gminas += 1
        
        if n_gminas == 0:
            continue
        
        # Compare with voivodeship pop
        voiv_pop = voiv_rec.pop.get(pd.Timestamp(yr, 1, 1), np.nan)
        agg_pop = agg[og_age, og_sex]
        
        if not np.isnan(voiv_pop) and voiv_pop > 0:
            pct_err = abs(agg_pop - voiv_pop) / voiv_pop * 100
            if pct_err > 1.0:
                errors.append({'voiv': voiv_tid, 'name': voiv_rec.name, 
                               'year': yr, 'agg': agg_pop, 'expected': voiv_pop,
                               'pct_err': pct_err, 'n_gminas': n_gminas, 'n_skip': n_skipped})

if errors:
    err_df = pd.DataFrame(errors).sort_values('pct_err', ascending=False)
    print(f"\nM_age_sex aggregation errors > 1%: {len(err_df)}")
    print(err_df.head(20).to_string(index=False))
else:
    print("\n✓ All voivodeship aggregations within 1% of expected population")

M_age_sex: shape=(16, 3), ogółem indices: age=15, sex=2
Labels n1: ['0-4', '10-14', '15-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '5-9', '50-54', '55-59', '60-64', '65-69', '70 i więcej', 'ogółem']

Checking 16 new voivodeships × years 1999-2024...

M_age_sex aggregation errors > 1%: 3
   voiv        name  year       agg  expected   pct_err  n_gminas  n_skip
1400000 MAZOWIECKIE  1999 6789968.0 5112652.0 32.807162       326       0
1400000 MAZOWIECKIE  2000 6787428.0 5115010.0 32.696280       326       0
1400000 MAZOWIECKIE  2001 6793408.0 5121681.0 32.640202       326       0


## 7. M_pop__age_sex vs M_age_sex Comparison

In [8]:
# ── Cell 8: Compare M_pop__age_sex and M_age_sex ──
# Show the label difference that causes the 19 vs 16 bin issue

# Get labels from both
for rec in db._records.values():
    mpas = rec.cross_tables.get('M_pop__age_sex')
    mas = rec.cross_tables.get('M_age_sex')
    if mpas and mpas.shape == (19, 3):
        mpas_labels = mpas.dim_labels
        break

for rec in db._records.values():
    mas = rec.cross_tables.get('M_age_sex')
    if mas:
        mas_labels = mas.dim_labels
        break

print("M_pop__age_sex (19×3) age labels:")
for i, l in enumerate(mpas_labels['n1']):
    in_mas = '✓' if l in mas_labels['n1'] else '✗ EXCLUDED'
    print(f"  [{i:2d}] {l:<20} {in_mas}")

print(f"\nM_age_sex (16×3) age labels:")
for i, l in enumerate(mas_labels['n1']):
    print(f"  [{i:2d}] {l}")

print(f"\n⚠ The 3 EXCLUDED labels in M_pop__age_sex overlap with other bins:")
excluded = set(mpas_labels['n1']) - set(mas_labels['n1'])
for l in sorted(excluded):
    print(f"  - '{l}'")
print(f"  These cause shape (19,3) vs (16,3) inconsistency across records.")
print(f"  M_age_sex has CONSISTENT (16,3) shape for ALL its 4184 records.")
print(f"  The estimator should use M_age_sex, NOT M_pop__age_sex.")

M_pop__age_sex (19×3) age labels:
  [ 0] 0-4                  ✓
  [ 1] 10-14                ✓
  [ 2] 15-19                ✓
  [ 3] 20-24                ✓
  [ 4] 25-29                ✓
  [ 5] 30-34                ✓
  [ 6] 35-39                ✓
  [ 7] 40-44                ✓
  [ 8] 45-49                ✓
  [ 9] 5-9                  ✓
  [10] 50-54                ✓
  [11] 55-59                ✓
  [12] 60-64                ✓
  [13] 65-69                ✓
  [14] 70-74                ✗ EXCLUDED
  [15] 75-79                ✗ EXCLUDED
  [16] 80-84                ✗ EXCLUDED
  [17] 85 i więcej          ✗ EXCLUDED
  [18] ogółem               ✓

M_age_sex (16×3) age labels:
  [ 0] 0-4
  [ 1] 10-14
  [ 2] 15-19
  [ 3] 20-24
  [ 4] 25-29
  [ 5] 30-34
  [ 6] 35-39
  [ 7] 40-44
  [ 8] 45-49
  [ 9] 5-9
  [10] 50-54
  [11] 55-59
  [12] 60-64
  [13] 65-69
  [14] 70 i więcej
  [15] ogółem

⚠ The 3 EXCLUDED labels in M_pop__age_sex overlap with other bins:
  - '70-74'
  - '75-79'
  - '80-84'
  - '85 i więce

## 8. M_hh_size_2000 Missing Labels Investigation

In [9]:
# ── Cell 9: Investigate M_hh_size_2000 records missing '3-osobowe' ──
# These 231 records have shape (5,) — which source data do they have?

hh_bad = []
hh_bad_sources = Counter()
for rec in db._records.values():
    ct = rec.cross_tables.get('M_hh_size_2000')
    if ct and ct.shape == (5,):
        # Check what raw census data this record has
        has_P2871 = bool(rec.get_data_by_subject('P2871'))
        has_P3420 = bool(rec.get_data_by_subject('P3420'))
        has_P4287 = bool(rec.get_data_by_subject('P4287'))
        hh_bad.append((rec.teryt_id, rec.name, rec.level, has_P2871, has_P3420, has_P4287))
        key = f"P2871={'Y' if has_P2871 else 'N'} P3420={'Y' if has_P3420 else 'N'} P4287={'Y' if has_P4287 else 'N'}"
        hh_bad_sources[key] += 1

print(f"M_hh_size_2000 records with (5,) shape: {len(hh_bad)}")
print(f"\nSource data breakdown:")
for key, cnt in hh_bad_sources.most_common():
    print(f"  {key}: {cnt} records")

# Check: for a bad record, what does P4287 look like?
if hh_bad:
    sample_tid = hh_bad[0][0]
    sample_rec = db.get_by_teryt_id(sample_tid)
    print(f"\nSample bad record: {sample_tid} ({sample_rec.name})")
    p4287_data = sample_rec.get_data_by_subject('P4287')
    if p4287_data:
        print(f"  P4287 DataSeries count: {len(p4287_data)}")
        for key, ds in sorted(p4287_data.items()):
            cats = ds.categories
            vals = [(y, ds.values.iloc[i]) for i, y in enumerate(range(1986, 2026)) if not np.isnan(ds.values.iloc[i])]
            print(f"    {cats} → {vals[:3]}")
    else:
        print(f"  No P4287 data")
    # Check P2871
    p2871_data = sample_rec.get_data_by_subject('P2871')
    if p2871_data:
        print(f"  P2871 labels:")
        for key, ds in sorted(p2871_data.items()):
            cats = ds.categories
            print(f"    {cats}")

# Root cause hypothesis: P4287 has '3-osobowe' but P2871 doesn't
# If only P4287 data is available (2021) and it has 3-osobowe, build_cross_table
# should produce (6,). But if P2871 (2002) was loaded first with only 5 labels
# then build_cross_table uses P2871's labels which don't include 3-osobowe.

M_hh_size_2000 records with (5,) shape: 0

Source data breakdown:


## 9. Summary of All Issues Found

In [10]:
# ── Cell 10: Summary ──
print("=" * 80)
print("COMPREHENSIVE DATA AUDIT — SUMMARY OF ISSUES")
print("=" * 80)

issues = [
    {
        'id': 1,
        'severity': 'CRITICAL',
        'title': 'M_age_sex missing for new voivodeships',
        'detail': ('_create_custom_merged_subjects() gates P2137 extraction by '
                   '`record.level == LEVEL_GMINA`, excluding voivodeship-level '
                   'P2137 data. All 16+2 new voivodeships lack M_age_sex. '
                   'Only old voivodeships (49) get it from H_age_sex.'),
        'impact': 'No voivodeship-level reference data for age×sex estimation constraints.',
        'fix': 'Remove level filter for P2137 in M_age_sex creation; add handling for powiats and voivodeships.'
    },
    {
        'id': 2,
        'severity': 'CRITICAL',
        'title': 'Warsaw (1431) double-counting in hierarchy',
        'detail': ('Both Warszawa (1431001, rdz=1) and its 11 districts (1431011-1431181, '
                   'all rdz=1) are listed as children of powiat 1431000. Warszawa pop = '
                   'sum of district pops. Aggregating rodz 1/2/3 produces EXACTLY 2× powiat pop.'),
        'impact': 'Mazowieckie aggregation errors (~33%), all estimations biased for Mazowieckie.',
        'fix': 'Exclude 1431001 from aggregation when 1431011-1431181 exist (or vice versa). '
               'Detect parent-child overlaps in rodz=1 by checking if any record has pop = sum of siblings.'
    },
    {
        'id': 3,
        'severity': 'HIGH',
        'title': 'estimate_age_sex_2000 uses M_pop__age_sex (wrong source)',
        'detail': ('The estimator uses M_pop__age_sex (19 labels, shapes (19,3) and (16,3)) '
                   'instead of M_age_sex (16 labels, consistent (16,3)). This causes 78 records '
                   'to be skipped due to shape mismatch and produces E_age_sex_2000 with wrong dimensionality.'),
        'impact': '53 gminas skipped, includes all Warsaw sub-units. E_age_sex_2000 shape (19,3) is wrong.',
        'fix': 'Change source to M_age_sex in ANCHOR_SUBJECTS config and _estimate_age_sex_2000().'
    },
    {
        'id': 4,
        'severity': 'MEDIUM',
        'title': 'M_hh_size_2000 inconsistent shapes: 231 records with (5,) vs (6,)',
        'detail': ('231 records missing "3-osobowe" label. These have only P4287 (2021) data '
                   'but build_cross_table produced (5,) — likely because P2871 was loaded with '
                   'different label structure.'),
        'impact': 'Cannot use M_hh_size_2000 consistently for all gminas in estimation.',
        'fix': 'Ensure _create_custom_merged_subjects stores ALL 6 labels, using NaN for missing.'
    },
    {
        'id': 5,
        'severity': 'LOW',
        'title': 'M_educ_1990 inconsistent shape: 1 record (Lesko) with (5,) vs (6,)',
        'detail': ('Lesko (1801044) missing "podstawowe nieukończone i bez wykształcenia". '
                   'Residual computation failed (ogółem unavailable or sum >= ogółem).'),
        'impact': 'Minor — only 1 record affected.',
        'fix': 'Same fix as issue 4 — store all labels with NaN for missing.'
    },
    {
        'id': 6,
        'severity': 'MEDIUM',
        'title': 'M_educ_2000 missing for new voivodeships in cross_table form',
        'detail': ('M_educ_2000 has data at levels [0, 2, 5, 6] but needs investigation '
                   'whether P2350/P4092 voivodeship data is properly stored.'),
        'impact': 'May affect voivodeship-level constraints for education estimation.',
        'fix': 'Verify P2350/P4092 BDL fill pass in _create_custom_merged_subjects.'
    },
]

for issue in issues:
    sev_color = {'CRITICAL': '🔴', 'HIGH': '🟠', 'MEDIUM': '🟡', 'LOW': '🟢'}[issue['severity']]
    print(f"\n{sev_color} Issue #{issue['id']}: [{issue['severity']}] {issue['title']}")
    print(f"   Detail: {issue['detail']}")
    print(f"   Impact: {issue['impact']}")
    print(f"   Fix: {issue['fix']}")

COMPREHENSIVE DATA AUDIT — SUMMARY OF ISSUES

🔴 Issue #1: [CRITICAL] M_age_sex missing for new voivodeships
   Detail: _create_custom_merged_subjects() gates P2137 extraction by `record.level == LEVEL_GMINA`, excluding voivodeship-level P2137 data. All 16+2 new voivodeships lack M_age_sex. Only old voivodeships (49) get it from H_age_sex.
   Impact: No voivodeship-level reference data for age×sex estimation constraints.
   Fix: Remove level filter for P2137 in M_age_sex creation; add handling for powiats and voivodeships.

🔴 Issue #2: [CRITICAL] Warsaw (1431) double-counting in hierarchy
   Detail: Both Warszawa (1431001, rdz=1) and its 11 districts (1431011-1431181, all rdz=1) are listed as children of powiat 1431000. Warszawa pop = sum of district pops. Aggregating rodz 1/2/3 produces EXACTLY 2× powiat pop.
   Impact: Mazowieckie aggregation errors (~33%), all estimations biased for Mazowieckie.
   Fix: Exclude 1431001 from aggregation when 1431011-1431181 exist (or vice versa). Dete

## 10. Post-Fix Verification

In [12]:
# ── Cell 11: Comprehensive Post-Fix Verification ──
print("="*80)
print("POST-FIX VERIFICATION — All 6 Issues")
print("="*80)

# Issue #1: M_age_sex voivodeship gap
new_voi_tids = [r.teryt_id for r in db._records.values()
                if r.level == LEVEL_VOIVODESHIP and int(r.teryt_id) < 5000000]
standard_voi = [t for t in new_voi_tids if t not in ('1300000', '1500000')]  # exclude NUTS-2 subregions
n_with_age_sex = sum(1 for t in standard_voi if db.get_by_teryt_id(t).cross_tables.get('M_age_sex'))
status1 = "✅ FIXED" if n_with_age_sex == len(standard_voi) else "❌ NOT FIXED"
print(f"\nIssue #1 — M_age_sex voivodeship gap: {status1}")
print(f"  {n_with_age_sex}/{len(standard_voi)} standard new voivodeships have M_age_sex")
# Also check levels available
levels_with_age_sex = set()
for r in db._records.values():
    if r.cross_tables.get('M_age_sex'):
        levels_with_age_sex.add(r.level)
print(f"  M_age_sex now available at levels: {sorted(levels_with_age_sex)}")

# Issue #2: Warsaw double-counting (check in estimator context)
from collections import Counter
# Check fix is deployed: filter_aggregation_children exists
import importlib
import geoTERYT_db as gtdb
has_filter = hasattr(gtdb, 'filter_aggregation_children')
status2 = "✅ FIXED" if has_filter else "❌ NOT FIXED"
print(f"\nIssue #2 — Warsaw double-counting: {status2}")
print(f"  filter_aggregation_children() exists: {has_filter}")
if has_filter:
    # Test it on Warsaw
    powiat_waw = db.get_by_teryt_id('1431000')
    raw_children = powiat_waw.get_children(2000)
    filtered = gtdb.filter_aggregation_children(raw_children, 2000, db._records)
    n_excluded = len(raw_children) - len(filtered)
    excluded_tids = set(raw_children) - set(filtered)
    print(f"  Warsaw 2000: {len(raw_children)} raw children → {len(filtered)} after filter")
    print(f"  Excluded: {excluded_tids}")
    # Verify sum after filter
    filt_pop = sum(db._records[t].pop.get(pd.Timestamp(f'2000-01-01'), 0) for t in filtered
                   if t in db._records and hasattr(db._records[t], 'pop'))
    print(f"  Filtered children pop sum: {filt_pop:.0f}")
    print(f"  Powiat pop: {powiat_waw.pop.get(pd.Timestamp('2000-01-01'), 0):.0f}")

# Issue #3: Estimator uses M_age_sex (not M_pop__age_sex)
import demographic_estimator as de
importlib.reload(de)
anchor_cfg = de.ANCHOR_SUBJECTS.get('age_sex', {}).get('2000', {})
uses_correct = anchor_cfg.get('anchor_subjects') == ['M_age_sex'] and \
               anchor_cfg.get('marginal_subjects') == ['M_age_sex']
status3 = "✅ FIXED" if uses_correct else "❌ NOT FIXED"
print(f"\nIssue #3 — Estimator source subject: {status3}")
print(f"  ANCHOR_SUBJECTS['age_sex']['2000'] = {anchor_cfg}")

# Issue #4: M_hh_size_2000 consistent shapes
hh_shapes = Counter()
for r in db._records.values():
    ct = r.cross_tables.get('M_hh_size_2000')
    if ct:
        hh_shapes[ct.shape] += 1
all_shape_6 = len(hh_shapes) == 1 and (6,) in hh_shapes
status4 = "✅ FIXED" if all_shape_6 else "❌ NOT FIXED"
print(f"\nIssue #4 — M_hh_size_2000 shapes: {status4}")
print(f"  Shapes: {dict(hh_shapes)}")

# Issue #5: M_educ_1990 consistent shapes
educ_shapes = Counter()
for r in db._records.values():
    ct = r.cross_tables.get('M_educ_1990')
    if ct:
        educ_shapes[ct.shape] += 1
all_shape_6e = len(educ_shapes) == 1 and (6,) in educ_shapes
status5 = "✅ FIXED" if all_shape_6e else "❌ NOT FIXED"
print(f"\nIssue #5 — M_educ_1990 shapes: {status5}")
print(f"  Shapes: {dict(educ_shapes)}")

# Issue #6: M_educ_2000 voivodeship coverage
educ2000_voiv = sum(1 for r in db._records.values()
                    if r.level == LEVEL_VOIVODESHIP and r.cross_tables.get('M_educ_2000')
                    and int(r.teryt_id) < 5000000)
status6 = "✅ OK" if educ2000_voiv >= 16 else "⚠ CHECK"
print(f"\nIssue #6 — M_educ_2000 voivodeship coverage: {status6}")
print(f"  {educ2000_voiv} new voivodeships with M_educ_2000 cross tables")
educ2000_levels = set()
for r in db._records.values():
    if r.cross_tables.get('M_educ_2000'):
        educ2000_levels.add(r.level)
print(f"  M_educ_2000 available at levels: {sorted(educ2000_levels)}")

# Final tally
all_fixed = all([status1.startswith("✅"), status2.startswith("✅"),
                 status3.startswith("✅"), status4.startswith("✅"),
                 status5.startswith("✅"), status6.startswith("✅")])
print(f"\n{'='*80}")
if all_fixed:
    print("ALL 6 ISSUES RESOLVED ✅")
else:
    print("SOME ISSUES REMAIN — see details above")
print("="*80)

POST-FIX VERIFICATION — All 6 Issues

Issue #1 — M_age_sex voivodeship gap: ✅ FIXED
  16/16 standard new voivodeships have M_age_sex
  M_age_sex now available at levels: [0, 2, 5, 6]

Issue #2 — Warsaw double-counting: ✅ FIXED
  filter_aggregation_children() exists: True
  Warsaw 2000: 12 raw children → 11 after filter
  Excluded: {'1431001'}
  Filtered children pop sum: 1672418
  Powiat pop: 1672418

Issue #3 — Estimator source subject: ✅ FIXED
  ANCHOR_SUBJECTS['age_sex']['2000'] = {'anchor_subjects': ['M_age_sex'], 'marginal_subjects': ['M_age_sex']}

Issue #4 — M_hh_size_2000 shapes: ✅ FIXED
  Shapes: {(6,): 4257}

Issue #5 — M_educ_1990 shapes: ✅ FIXED
  Shapes: {(6,): 3725}

Issue #6 — M_educ_2000 voivodeship coverage: ✅ OK
  18 new voivodeships with M_educ_2000 cross tables
  M_educ_2000 available at levels: [0, 2, 5, 6]

ALL 6 ISSUES RESOLVED ✅
